# Jacobian Lens Demo

| 章節 | 內容 |
|---|---|
| 1. 架構 | Qwen 3.6 27b VLM 架構 |
| 2. 原理 | 語言模型運作過程 |
| 3. 實作 | 以 `The capital of France is` 為範例 |
| 4. 介入 | Concept swap：將 `France` 換成 `China` |

## 1. Qwen3.6-27B VLM 架構

影像與文字先轉成相同維度的向量，再合併成一條 sequence。

Decoder 共 64 層：每 3 層 GatedDeltaNet，接 1 層 Attention，重複 16 次。

最後一個位置的向量先過 RMSNorm；`lm_head` 將 5120 維投影成 248,320 個 logits，`softmax` 再轉成下一個 token 的機率。

![Qwen3.6-27B VLM](assets/qwen3_6_27b_vlm_flow.png)

## 2. 語言模型如何運作

<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 32px; align-items: start;">
  <div>
    <p>每個 token 在每一層都有一個 5120 維向量 <i>h</i><sub>ℓ</sub>，稱為 residual stream。</p>
    <p><code>h ← h + Mixer(RMSNorm(h))</code><br>
    <code>h ← h + FFN(RMSNorm(h))</code></p>
    <p><b>為什麼研究它？</b> Residual stream 是 decoder 的共享通道：</p>
    <ul>
      <li>每一層都從它讀取資訊，也把結果寫回去。</li>
      <li>下一層與最後的 logits 都由它決定。</li>
      <li>修改 <i>h</i><sub>ℓ</sub> 後繼續 forward，可以直接檢驗因果。</li>
    </ul>
    <p>所以，讀 <i>h</i><sub>ℓ</sub> 是看模型算到哪；改 <i>h</i><sub>ℓ</sub> 是測模型有沒有使用那個資訊。Logit lens 與 Jacobian lens 都是在讀它。</p>
  </div>
  <img src="assets/qwen3_6_27b_arch_residual.png" alt="Residual stream" style="width: 100%; height: auto;">
</div>

## 3. 實作

### 3.1 Forward pass

In [ ]:
import mlx.core as mx
from mlx_vlm import load

model, processor = load("models/Qwen3.6-27B-4bit")
tokenizer = processor.tokenizer
language_model = model.language_model
decoder = language_model.model

In [ ]:
# text → token ids
text = "The capital of France is"
token_ids = mx.array(tokenizer.encode(text))[None]
print(token_ids)

In [ ]:
# token ids → embedding vectors
embeddings = decoder.embed_tokens(token_ids)
print(embeddings.shape)                # batch × tokens × dimensions
print(embeddings[0, 0, :8].tolist())  # first 8 values for "The"

In [ ]:
# embedding vectors → logits
logits = language_model(token_ids, inputs_embeds=embeddings).logits
print(logits.shape)                    # batch × positions × vocabulary
print(logits[0, -1, :8].tolist())     # first 8 logits at the last position

In [ ]:
# logits → next token id
next_token_id = int(mx.argmax(logits[0, -1]))
print(next_token_id)

In [ ]:
# next token id → next token
next_token = tokenizer.decode([next_token_id])
print(repr(next_token))

### 3.2 Residual stream

一般 forward pass 只回傳 logits。現在保留每一層的 residual stream。

In [ ]:
# keep the residual stream after every decoder layer
layers = range(len(decoder.layers))
output = language_model(token_ids, capture_layer_ids=layers)
residuals = mx.stack(output.hidden_states)

print(residuals.shape)  # layers × batch × tokens × dimensions

In [ ]:
# last token position across all layers
h = residuals[:, 0, -1]

print(h.shape)  # layers × dimensions

### 3.3 Logit lens

Logit lens 借用模型最後的 RMSNorm 與 `lm_head`，直接讀取中間層的 residual stream：

$$
\operatorname{LogitLens}(h_\ell) = W_U\,\operatorname{RMSNorm}(h_\ell)
$$

這等於假設後面的 layers 不再改變表示。

In [ ]:
# apply the model's final readout to every intermediate residual
logit_lens_logits = language_model.lm_head(decoder.norm(h))

print(logit_lens_logits.shape)  # layers × vocabulary

In [ ]:
# inspect layer 55
layer = 55
paris_id = tokenizer.encode(" Paris")[0]  # leading space is part of the token
scores = logit_lens_logits[layer]

top_token = tokenizer.decode([int(mx.argmax(scores))])
paris_probability = float(mx.softmax(scores)[paris_id])

print(f"L{layer}: top token = {top_token!r}")
print(f"prodability (' Paris') = {paris_probability:.3f}")

### 3.4 Jacobian lens

#### Logit lens 的缺陷

Logit lens 把中間層的 $h_\ell$ 直接交給只用在最終層的 `lm_head`，等於假設剩下的 layers 什麼都不做。

但後面的 layers 仍會旋轉、縮放與混合表示。資訊可能已經存在，只是還沒轉成 `lm_head` 看得懂的方向。

#### Jacobian 是什麼？

中間層的資訊還要經過後續 layers，才會變成最終輸出。Anthropic 的思路不是直接解碼 $h_\ell$，而是先近似「剩下的 layers 會如何轉換它」。

因此先問一個更小的問題：如果 $h_\ell$ 改變一點，$h_{\mathrm{final}}$ 會跟著怎麼變？描述這種局部變化的工具就是 Jacobian。

對一維函數 $y=f(x)$，導數 $f'(x)$ 回答：「$x$ 動一點，$y$ 會動多少？」

Residual stream 是向量，所以導數變成矩陣：

$$
\Delta h_{\mathrm{final}} \approx J_\ell\,\Delta h_\ell
$$

$J_\ell[i,j]$ 表示：$h_\ell$ 的第 $j$ 維動一點，最終 residual 的第 $i$ 維會改變多少。

#### J-lens 的核心

把 $J_\ell$ 當成中間層到最終層的座標轉換，再交給同一個 output head：

$$
h_\ell \xrightarrow{\;J_\ell\;} J_\ell h_\ell
\xrightarrow{\;\mathrm{RMSNorm}+W_U\;} \text{logits}
$$

如果上過深度學習，這一步就是熟悉的 backpropagation。Backprop 不會把整張 Jacobian 展開；它從 output 端的一個方向 $u$ 開始，逐層往回傳，直接算出 $J_\ell^{\top}u$。

在 J-lens 裡，$u=w_t\odot\gamma$：$w_t$ 是 token $t$ 在 `lm_head` 裡的方向，$\gamma$ 是 final RMSNorm 的權重。因此，要得到一個 token 的 J-lens direction，只需要一次 backward pass。

實際使用的 $J_\ell$ 會跨 prompts 與 positions 平均，保留模型普遍的轉換方式，而不是只記住一句輸入。

#### 以 `The capital of France is` 為範例

對 prompt `The capital of France is`，我們要找出：L55 往哪個 5120 維方向移動，最終會讓模型更傾向輸出 `' Paris'`？

$$
\delta_{55}\ \longrightarrow\ \mathrm{L56\text{--}L63}\ \longrightarrow\ h_{\mathrm{final}}\ \longrightarrow\ \text{Paris score}
$$

最後對 Paris score backward，就能得到這個方向。

##### 1. 從 L55 跑到最終層

在 L55 residual 加上可微分的 $\delta$，再跑完 L56–L63。這讓 autograd 能追蹤：L55 改一點，最終 residual 會怎麼改？

In [ ]:
# 在 L55 加入可微分的 delta，只重跑尚未完成的 L56–L63
from mlx_vlm.models.qwen3_5.language import _create_qwen3_5_attention_mask

source = residuals[layer]  # batch × 5 tokens × 5120 dimensions
attention_mask = _create_qwen3_5_attention_mask(source, None)

# Qwen MRoPE 有三個位置軸；純文字的三軸都使用相同 token 順序
token_count = source.shape[1]
token_positions = mx.arange(token_count)
position_ids = mx.broadcast_to(token_positions, (3, 1, token_count))

remaining_blocks = decoder.layers[layer + 1:]

def remaining_layers(delta):
    x = source + delta
    for block in remaining_blocks:
        mask = None if block.is_linear else attention_mask
        x = block(x, mask=mask, position_ids=position_ids)
    return x

##### 2. 取出 `' Paris'` readout

VJP（vector–Jacobian product）就是「從指定 output 做一次 backward」。這裡的函數把 5120 維 final residual 轉成 248,320 個 logits；`paris_logit_selector` 指定只從 `' Paris'` logit 往回傳。

因此不必建立 $248{,}320\times5120$ 的 Jacobian，就能直接得到 `' Paris'` 對 final residual 的梯度：

$$u_{\text{Paris}}=w_{\text{Paris}}\odot\gamma\in\mathbb{R}^{5120}$$

In [ ]:
# One-hot selector：backward 時只關心 248,320 個 logits 中的 " Paris"
vocab_size = model.config.text_config.vocab_size
paris_logit_selector = mx.zeros((vocab_size,), dtype=h.dtype)
paris_logit_selector[paris_id] = 1

def final_gain_and_lm_head(final_residual):
    scaled_residual = final_residual * decoder.norm.weight
    return language_model.lm_head(scaled_residual)

# VJP：從 " Paris" logit backward 到 final residual
zero_residual = mx.zeros((h.shape[-1],), dtype=h.dtype)
_, (paris_readout,) = mx.vjp(
    final_gain_and_lm_head,
    (zero_residual,),
    (paris_logit_selector,),
)

print(paris_readout.shape)

##### 3. 反向傳播回 L55

先把 final residual 壓成一個 Paris score，再對 $\delta$ 求梯度：

$$\nabla_\delta\,\mathrm{score}=J_{55}^{\top}(w_{\text{Paris}}\odot\gamma)$$

Backprop 直接算這個向量，不必建立含 2,621 萬個數的完整 $J_{55}$。

In [ ]:
# 將最終 residual 投影成一個 " Paris" score
def paris_score(delta):
    return remaining_layers(delta)[0, -1] @ paris_readout

# train(True) 只切換到可微分的 GatedDeltaNet；不會更新 weights
for block in remaining_blocks:
    block.train(True)

# " Paris" score 對 L55 最後一個 token residual 的 gradient
# 在 delta=0 backward：J₅₅ᵀ (w_Paris ⊙ gamma)
local_paris_direction = mx.grad(paris_score)(mx.zeros_like(source))[0, -1]
mx.eval(local_paris_direction)  # MLX 是 lazy evaluation

for block in remaining_blocks:
    block.train(False)

print(local_paris_direction.shape)

##### `(5120,)` 與 `(5120, 5120)` 的差別

剛才先指定 `' Paris'`，所以只算出它需要的一個方向：

$$v_{\text{Paris}}=J_{55}^{\top}u_{\text{Paris}}\in\mathbb{R}^{5120}$$

這個 `(5120,)` 不是完整 $J_{55}$。若要保留 final residual 的所有 5120 個 output directions，就要分別用 $e_1,\ldots,e_{5120}$ 做 VJP，再把結果疊成：

$$J_{55}\in\mathbb{R}^{5120\times5120}$$

#### 為什麼需要很多 prompts？

剛才的 direction 只對這一句、L55、最後一個位置成立。換一個 prompt，attention 與 gates 改變，Jacobian 也會改變。

Anthropic 因此對許多 prompts、來源位置與未來輸出位置取平均：

$$
\bar J_\ell = \mathbb{E}_{\text{prompt},\,t,\,t'\ge t}\left[J_{\ell,t\to t'}\right]
$$

平均會削弱單一情境的路由細節，留下跨情境都穩定的「這個方向如何影響未來語言輸出」。

##### 載入平均後的 J-lens

下面使用 [Neuronpedia 的 Qwen3.6-27B lens](https://huggingface.co/neuronpedia/jacobian-lens/blob/main/qwen3.6-27b/jlens/Salesforce-wikitext/Qwen3.6-27B_jacobian_lens_n1000.pt)：1000 個 prompts 擬合的 `(5120, 5120)` 矩陣；本機檔案已轉成 MLX 的 `.npz` 格式。

In [ ]:
# use the averaged Jacobian for the full-vocabulary readout
lens = mx.load("models/Qwen3.6-27B-4bit/jlens.npz")
J = lens[f"J_{layer}"]
prompt_count = int(lens["__n_prompts__"].tolist()[0])
transported = (h[layer].astype(J.dtype) @ J.T).astype(h.dtype)

print(f"{prompt_count} prompts → {J.shape}")

In [ ]:
# read the transported residual with the same output head
jacobian_lens_logits = language_model.lm_head(decoder.norm(transported))
jacobian_top_token = tokenizer.decode([int(mx.argmax(jacobian_lens_logits))])
jacobian_probability = float(mx.softmax(jacobian_lens_logits)[paris_id])

print(f"L{layer}: top token = {jacobian_top_token!r}")
print(f"p(' Paris') = {jacobian_probability:.3f}")

### 3.5 Logit lens vs Jacobian lens

兩種 lens 讀取同一批 residual。差別是 Jacobian lens 先用 $J_\ell$ 補上後續 layers 的局部轉換。

下面追蹤每一層的 `p(' Paris')`。越早升高，代表越早讀到最終答案。

In [ ]:
import matplotlib.pyplot as plt

released_layers = sorted(int(x) for x in lens["__source_layers__"].tolist())
logit_curve, jacobian_curve, top_tokens = [], [], []

for layer_index in released_layers:
    residual = h[layer_index]
    logit_scores = language_model.lm_head(decoder.norm(residual))

    J = lens[f"J_{layer_index}"]
    transported = (residual.astype(J.dtype) @ J.T).astype(h.dtype)
    jacobian_scores = language_model.lm_head(decoder.norm(transported))

    logit_curve.append(float(mx.softmax(logit_scores)[paris_id]))
    jacobian_curve.append(float(mx.softmax(jacobian_scores)[paris_id]))
    top_tokens.append((
        layer_index,
        tokenizer.decode([int(mx.argmax(logit_scores))]),
        tokenizer.decode([int(mx.argmax(jacobian_scores))]),
    ))

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(released_layers, logit_curve, "o-", label="Logit lens", color="#D97757")
ax.plot(released_layers, jacobian_curve, "o-", label="Jacobian lens", color="#3B82C4")
ax.set(xlabel="layer", ylabel="p(' Paris')")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(alpha=0.2)
ax.legend(frameon=False)
plt.show()

print(f"{'layer':>5}  {'Logit lens':<16}  {'Jacobian lens'}")
for layer_index, logit_token, jacobian_token in top_tokens[::4]:
    print(f"L{layer_index:<4}  {logit_token!r:<16}  {jacobian_token!r}")

## 4. Concept swap

<div style="display: grid; grid-template-columns: 1.15fr 0.85fr; gap: 32px; align-items: center;">
  <div>
    <p>J-lens 讀到一個概念，不代表模型真的使用它。</p>
    <p>Concept swap 直接改寫 residual：若把 <code>France</code> 換成 <code>China</code>，答案也從 <code>Paris</code> 轉向 <code>Beijing</code>，才有因果證據。</p>
    <p>圖中的 <code>banana → elephant</code> 表示同一件事：換掉 J-space 裡的概念，再觀察後續輸出。</p>
  </div>
  <div>
    <img src="assets/jspace_concept_swap.png" alt="Concept swap in J-space" style="width: 100%; height: auto;">
    <p style="font-size: 0.8em; color: #777;">Source: <a href="https://www.anthropic.com/research/global-workspace">Anthropic, The global workspace of a large language model</a></p>
  </div>
</div>

### 4.1 交換 J-space 座標

每個 token 在第 $\ell$ 層都有一個 J-lens direction：

$$v_t=(W_UJ_\ell)_t$$

把 `France` 與 `China` 的 directions 排成 $V$，先測量 residual 在兩個方向上的座標 $c$，再交換它們：

$$
V=[v_{\text{France}}\;v_{\text{China}}],\qquad
c=V^\dagger h,\qquad
h'=h+\alpha V\bigl(\operatorname{swap}(c)-c\bigr)
$$

$V^\dagger$ 是 pseudoinverse。它能在兩個 directions 不垂直時，正確算出各自的座標。$\alpha=1$ 是交換；$\alpha>1$ 是 oversteer。

In [ ]:
# 取出 4-bit lm_head 中，某個 token 的 5120 維 readout
output_head = language_model.lm_head

def unembed_row(token_id):
    index = mx.array([token_id])
    return mx.dequantize(
        output_head.weight[index],
        output_head.scales[index],
        output_head.biases[index],
        group_size=output_head.group_size,
        bits=output_head.bits,
    )[0].astype(mx.float32)

def jspace_direction(token_id, layer_index):
    J = lens[f"J_{layer_index}"]
    direction = unembed_row(token_id).astype(J.dtype) @ J
    return direction.astype(mx.float32)

france_id = tokenizer.encode(" France")[0]
china_id = tokenizer.encode(" China")[0]

In [ ]:
# 只交換 residual 在 France–China 平面上的座標
def swap_coordinates(residual, france_direction, china_direction, strength=1.0):
    V = mx.stack([france_direction, china_direction], axis=1)
    coordinates = mx.linalg.pinv(V, stream=mx.cpu) @ residual
    swapped = coordinates[::-1]
    return residual + strength * (V @ (swapped - coordinates))

# 對照組：只加入 China，不移除 France
def add_china_only(residual, france_direction, china_direction, strength=1.0):
    V = mx.stack([france_direction, china_direction], axis=1)
    coordinates = mx.linalg.pinv(V, stream=mx.cpu) @ residual
    return residual + strength * (coordinates[0] - coordinates[1]) * china_direction

### 4.2 在 forward pass 中介入

在 L40–L55，每層處理完 `France` token 後交換一次座標。修改最後一個 token 容易讓模型直接輸出 `China`，而不是使用它回答問題。

In [ ]:
from mlx_vlm.models.qwen3_5.language import _create_qwen3_5_ssm_mask

swap_layers = [i for i in released_layers if 40 <= i <= 55]
model_dtype = decoder.norm.weight.dtype
france_directions = {i: jspace_direction(france_id, i) for i in swap_layers}
china_directions = {i: jspace_direction(china_id, i) for i in swap_layers}

def forward_with_swap(prompt, strength=0.0, positions=None, edit=swap_coordinates):
    ids = mx.array(tokenizer.encode(prompt))[None]
    x = decoder.embed_tokens(ids)

    attention_mask = _create_qwen3_5_attention_mask(x, None)
    state_space_mask = _create_qwen3_5_ssm_mask(x, None)
    token_positions = mx.arange(x.shape[1])
    position_ids = mx.broadcast_to(token_positions, (3, 1, x.shape[1]))

    if positions is None:
        pieces = [tokenizer.decode([int(t)]) for t in ids[0].tolist()]
        positions = [next(i for i, piece in enumerate(pieces) if "France" in piece)]

    for layer_index, block in enumerate(decoder.layers):
        mask = state_space_mask if block.is_linear else attention_mask
        x = block(x, mask=mask, position_ids=position_ids, position_embeddings=None)

        if strength and layer_index in swap_layers:
            france = france_directions[layer_index]
            china = china_directions[layer_index]
            for position in positions:
                x[0, position] = edit(
                    x[0, position].astype(mx.float32), france, china, strength
                ).astype(model_dtype)

    logits = language_model.lm_head(decoder.norm(x))[0, -1]
    return ids, logits

### 4.3 同一個 swap，四種問題

四個 prompts 使用相同 layers、位置規則與 strength。若 capital、language、continent、currency 一起改變，表示後續任務共用同一份 `France` 表示。

In [ ]:
tasks = [
    ("capital", "The capital of France is", " Paris", " Beijing"),
    ("language", "The language spoken in France is", " French", " Chinese"),
    ("continent", "The continent that contains France is", " Europe", " Asia"),
    ("currency", "The currency of France is called the", " Euro", " Yuan"),
]
conditions = [
    ("before", 0.0, swap_coordinates),
    ("swap α=1", 1.0, swap_coordinates),
    ("swap α=2", 2.0, swap_coordinates),
    ("add China", 2.0, add_china_only),
]

results = {}
for task, prompt, original, swapped in tasks:
    for condition, strength, edit in conditions:
        _, scores = forward_with_swap(prompt, strength, edit=edit)
        probabilities = mx.softmax(scores)
        results[task, condition] = (
            float(probabilities[tokenizer.encode(original)[0]]),
            float(probabilities[tokenizer.encode(swapped)[0]]),
            tokenizer.decode([int(mx.argmax(scores))]),
        )

每張小圖比較原答案與換入答案的機率。`add China` 若仍保留原答案，代表只加入 target direction 不等於完成概念交換。

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(11, 3), sharey=True)
for ax, (task, _, original, swapped) in zip(axes, tasks):
    original_p = [results[task, name][0] for name, _, _ in conditions]
    swapped_p = [results[task, name][1] for name, _, _ in conditions]
    x = list(range(len(conditions)))
    ax.bar([i - 0.18 for i in x], original_p, 0.36, label=original.strip(), color="#D97757")
    ax.bar([i + 0.18 for i in x], swapped_p, 0.36, label=swapped.strip(), color="#3B82C4")
    ax.set_title(task)
    ax.set_xticks(x, [name.replace(" ", "\n") for name, _, _ in conditions], fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8)
axes[0].set_ylabel("probability")
plt.tight_layout()
plt.show()

for task, prompt, _, _ in tasks:
    before = results[task, "before"][2]
    after = results[task, "swap α=2"][2]
    print(f"{task:10s} {before!r:12s} → {after!r}")

`swap α=1` 是嚴格交換；`swap α=2` 用來看較弱的方向是否需要 oversteer。`add China` 是對照組：若只加入 `China` 不夠，改變行為需要同時移除 `France`。

在這顆 Qwen 上，capital、language、currency 的方向較清楚；continent 較弱。

### 4.4 交換隱藏的中間答案

考慮：

```text
The capital of the country where the Eiffel Tower stands is
```

模型必須先得到 `Eiffel Tower → France`，再得到 `France → Paris`。`France` 沒出現在 prompt，因此把內部的 `France` 換成 `China` 後：

- 輸出 `China`：介入的概念直接洩漏到輸出。
- 輸出 `Beijing`：模型使用新概念完成了第二跳。

In [ ]:
two_hop_prompt = "The capital of the country where the Eiffel Tower stands is"
two_hop_ids = mx.array(tokenizer.encode(two_hop_prompt))[None]

# 不改最後一個位置，避免直接把 China 推到輸出
content_positions = list(range(1, two_hop_ids.shape[1] - 1))

watch = [" Paris", " Beijing", " France", " China"]
for strength in (0.0, 1.0, 2.0):
    _, scores = forward_with_swap(
        two_hop_prompt, strength, positions=content_positions
    )
    probabilities = mx.softmax(scores)
    top_token = tokenizer.decode([int(mx.argmax(scores))])
    watched = "  ".join(
        f"p({token.strip()})={float(probabilities[tokenizer.encode(token)[0]]):.3f}"
        for token in watch
    )
    print(f"α={strength:<3} top={top_token!r:12s} {watched}")

若 `Paris → Beijing`，同時 `France`、`China` 的輸出機率仍低，這次介入改變的是中間推理，而不是直接指定下一個 token。

Concept swap 仍是一階近似。結果也會受 layer、position 與 $\alpha$ 影響，因此需要 baseline、add-only 與 leakage checks。